# 06 · 건강검진 데이터 — 혈색소와 가장 관련 깊은 속성 찾기
교과서 **80~82쪽 활동**(Ⅱ-02 데이터 가공과 핵심 속성 추출)을 우리 수업 방식으로 옮겼습니다.

맨 먼저 **파일 → 드라이브에 사본 저장**을 하세요(내 드라이브에 저장돼야 고친 내용이 남습니다).
`____` 가 보이면 거기가 문제입니다. 빈칸 **바로 위 💡 힌트**를 읽고 채운 뒤 셀을 실행하세요(Shift + Enter).
교과서는 `files.upload()` 로 파일을 올리지만, 이 노트북은 **선생님 저장소에서 바로 읽어** 올릴 파일이 없습니다.

> 원본은 공공데이터포털 «국민건강보험공단_건강검진정보»(100만 명, 약 95MB)입니다. 코랩에서 바로 열리도록 **3만 명을 무작위로 뽑은 표본**을 씁니다.
> 결과 숫자가 교과서와 조금 다를 수 있지만, 무엇이 가장 관련 깊은지는 같게 나옵니다.

## 단계 1 · 데이터 불러오기

In [ ]:
주소 = 'https://raw.githubusercontent.com/richee-pc/AI_cs/main/data/'
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(주소 + 'health_checkup_30k.csv')
df.shape

## 단계 2 · 데이터 살펴보기
💡 앞의 다섯 행은 `head()`, 열마다 자료형과 빈칸 수는 `info()` 입니다.

In [ ]:
df.____()

In [ ]:
df.info()

## 단계 3 · 전처리
### ① 결측치 세기
💡 비어 있는 칸을 찾는 함수는 `isnull()`, 열마다 세려면 `.sum()` 을 이어 붙입니다.

In [ ]:
df.____().sum()

### ② 축소 — 필요 없는 열 빼기
- 건강과 직접 관련 없는 열: `기준년도`, `가입자일련번호`, `시도코드`
- 통째로 비어 있는 열: `결손치유무`, `치아마모증율`, `제3대구치(사랑니)이상`
- 상관계수를 구하기 어려운 **범주형** 열: `성별`, `흡연상태`, `음주여부`, `구강검진수검여부`, `치아우식증유무`, `치석`, `요단백`, `청력(좌)`, `청력(우)`

💡 열을 지울 때는 `axis=1`. 그리고 **`df = ` 로 다시 받아야** 지워진 채로 남습니다(가장 잦은 실수!).

In [ ]:
빼기 = ['기준년도', '가입자일련번호', '시도코드',
        '결손치유무', '치아마모증율', '제3대구치(사랑니)이상',
        '성별', '흡연상태', '음주여부', '구강검진수검여부', '치아우식증유무', '치석', '요단백', '청력(좌)', '청력(우)']
df = df.drop(빼기, axis=____)
df.columns

### ③ 정제 — 이상치 확인
상자그림에서 상자 바깥 점이 이상치 후보입니다. 교과서는 연습용 기준으로 `연령대코드(5세 단위)` 가 **18 이상인 행**을 뺍니다.
(교과서도 «실제로는 충분히 생각해 정하라»고 합니다 — 정말 잘못된 값일까요?)

💡 남길 조건은 «18보다 작다» → `< 18`

In [ ]:
plt.boxplot(df['연령대코드(5세 단위)'])
plt.title('before')
plt.show()

print('전 :', len(df))
df = df[df['연령대코드(5세 단위)'] ____ 18]
print('후 :', len(df))

## 단계 4 · 상관계수로 핵심 속성 찾기
💡 정렬은 `sort_values()`. 큰 것부터 보려면 `ascending=False`.

In [ ]:
corr = df.corr()['혈색소']
print('— 함께 커지는 속성(양의 상관) —')
print(corr.sort_values(ascending=____).head(6))
print()
print('— 반대로 움직이는 속성(음의 상관) —')
print(corr.sort_values(ascending=True).head(5))

## 생각해 보기
1. 혈색소와 가장 관련 깊은 속성 두 개는? ( ______ , ______ )
2. «키가 클수록 혈색소가 높다» — 이것이 **원인과 결과**일까요? 함께 움직이게 만드는 다른 속성(예: 성별)은 없을까요?
3. 범주형인 `성별` 을 숫자(1·2)로 두면 상관계수를 구할 수는 있습니다. 그래도 빼야 할까요?

> 적어 보기:

---
### 정답 예시 (막혔을 때만)
- 단계 2 `df.head()` · 단계 3 ① `df.isnull().sum()` · ② `axis=1` · ③ `< 18` · 단계 4 `ascending=False`
- 교과서 결과처럼 **신장 · 체중 · 허리둘레**가 양의 상관, **HDL콜레스테롤 · 연령대**가 음의 상관으로 나옵니다.
- 신장과 혈색소는 둘 다 **성별**의 영향을 크게 받습니다. 남녀를 나눠 다시 상관을 구해 보면 관계가 크게 줄어듭니다 — 상관 ≠ 원인.